In [1]:
import pandas as pd
from tqdm import tqdm
from pyterrier_pisa import PisaIndex
import pyterrier as pt
if not pt.started():
    pt.init()

PyTerrier 0.10.0 has loaded Terrier 5.10 (built by craigm on 2024-08-22 17:33) and terrier-helper 0.0.8



In [2]:
training_file = pd.read_csv("./data/training_data_w_qids.csv")

In [3]:
with open("./data/sans_strat1_7neg.tsv", "w") as f:
    sensitive_docs = training_file[training_file["sensitivity"] == 1]
    for _, row in tqdm(training_file.iterrows(), total = len(training_file), desc = "Generating negs"):
        for_sample = sensitive_docs[sensitive_docs["medline_ui"] != row.medline_ui]
        negatives = for_sample.sample(n=7).medline_ui
        for negative in negatives:
            sample = row.qid, row.medline_ui, negative
            f.write(f"{sample[0]}\t{sample[1]}\t{sample[2]}\n")        

Generating negs:  13%|█▎        | 43376/334070 [02:59<20:04, 241.29it/s]


KeyboardInterrupt: 

In [ ]:
with open("./data/sans_strat2_7neg.tsv", "w") as f:
    sensitive_docs = training_file[training_file["sensitivity"] == 1]
    non_sensitive_docs = training_file[training_file["sensitivity"] == 0]
    for _, row in tqdm(training_file.iterrows(), total = len(training_file), desc = "Generating negs"):
        if row.sensitivity == 0:
            for_sample = training_file[training_file["medline_ui"] != row.medline_ui]
            negatives = for_sample.sample(n=7).medline_ui
        else:
            for_sample = sensitive_docs[sensitive_docs["medline_ui"] != row.medline_ui]
            negatives = for_sample.sample(n=7).medline_ui
            
        for negative in negatives:
                sample = row.qid, row.medline_ui, negative
                f.write(f"{sample[0]}\t{sample[1]}\t{sample[2]}\n")        

In [7]:
index_path = "/nfs/primary/scratch/classifier_scoring/pisa_index_v5"
training_file = training_file.rename(columns = {"medline_ui" : "docno", "abstract_title" : "text"})
training_file.docno = training_file.docno.astype(str)

iter_indexer = PisaIndex(index_path, text_field = ["text"])
indexref = iter_indexer.index(training_file.to_dict(orient="records"))
threads = 1024
index = PisaIndex(index_path, threads = threads)
model = index.bm25(verbose = True) % 100
model.search("Hello my name is Jack")

[2024-12-05 12:33:00.056] [info] [Batch 3] Processed documents [300000, 334070)
[2024-12-05 12:33:08.406] [info] [Batch 0] Processed documents [0, 100000)
[2024-12-05 12:33:08.883] [info] [Batch 1] Processed documents [100000, 200000)
[2024-12-05 12:33:09.677] [info] [Batch 2] Processed documents [200000, 300000)
[2024-12-05 12:33:10.644] [info] Merging titles
[2024-12-05 12:33:10.754] [info] Creating document lexicon
[2024-12-05 12:33:10.962] [info] Merging URLs
[2024-12-05 12:33:10.978] [info] Collecting terms
[2024-12-05 12:33:11.086] [info] Writing terms
[2024-12-05 12:33:11.119] [info] Mapping terms
[2024-12-05 12:33:11.134] [info] Remapping IDs
[2024-12-05 12:33:15.814] [info] Concatenating batches
[2024-12-05 12:33:15.927] [info] Success.
[2024-12-05 12:33:18.735] [info] Inverting [0, 100000)
[2024-12-05 12:33:21.410] [info] Inverting [100000, 200000)
[2024-12-05 12:33:24.184] [info] Inverting [200000, 300000)
[2024-12-05 12:33:26.823] [info] Inverting [300000, 334070)
[2024-12-

Storing terms statistics: 100% [0s]
Storing score upper bounds: 41% [0s]

[2024-12-05 12:33:32.359] [info] Dropping 0 terms
[2024-12-05 12:33:32.646] [info] Reading sizes...
[2024-12-05 12:33:32.649] [info] Storing max weight for each list and for each block...
[2024-12-05 12:33:34.740] [info] number of elements / number of blocks: 52.4672
[2024-12-05 12:33:34.826] [info] Processing 334070 documents (streaming)
/nfs/primary/scratch/classifier_scoring/pisa_index_v4/bm25.k1-0.9.b-0.4.q0.bmw.64.block_simdbp
[2024-12-05 12:33:35.415] [info] Index compressed in 0.589107 seconds


Storing score upper bounds: 100% [0s]
Create index: 100% [0s]   | 0/1 [00:00<?, ?query/s]index: 81% [0s]
PISA bm25: 100%|██████████| 1/1 [00:00<00:00,  1.47query/s]


,qid,query,docno,score,rank
0,1,Hello my name is Jack,87175621,14.950690,0
1,1,Hello my name is Jack,91221344,14.485616,1
2,1,Hello my name is Jack,88259345,13.922976,2
3,1,Hello my name is Jack,91024253,12.592087,3
4,1,Hello my name is Jack,91368239,12.532845,4
...,...,...,...,...,...
95,1,Hello my name is Jack,88154275,6.762294,95
96,1,Hello my name is Jack,88132188,6.757550,96
97,1,Hello my name is Jack,88132191,6.757550,97
98,1,Hello my name is Jack,88132195,6.757550,98


In [12]:
training_file = training_file.rename(columns = {"docno" : "medline_ui", "text" : "abstract_title"})

with open("./data/sans_strat3_7neg.tsv", "w") as f:
    sensitive_docs = training_file[training_file["sensitivity"] == 1]
    non_sensitive_docs = training_file[training_file["sensitivity"] == 0]
    for _, row in tqdm(training_file.iterrows(), total = len(training_file), desc = "Generating negs"):
        if row.sensitivity == 0:
            for_sample = training_file[training_file["medline_ui"] != row.medline_ui]
            negatives = for_sample.sample(n=7).medline_ui
        else:
            bm25_res = model.search(row.best_query)
            negatives = bm25_res.sample(n=7).docno
            
        for negative in negatives:
                sample = row.qid, row.medline_ui, negative
                print(sample)
                f.write(f"{sample[0]}\t{sample[1]}\t{sample[2]}\n")        

Generating negs:   0%|          | 3/334070 [00:00<8:05:57, 11.46it/s] 

(0, '88000001', '87238263')
(0, '88000001', '90123374')
(0, '88000001', '87297883')
(0, '88000001', '90154597')
(0, '88000001', '89155691')
(0, '88000001', '88320534')
(0, '88000001', '87115841')
(1, '88000002', '88143854')
(1, '88000002', '89100437')
(1, '88000002', '88212874')
(1, '88000002', '88025548')
(1, '88000002', '87200424')
(1, '88000002', '90201781')
(1, '88000002', '90327387')
(2, '88000003', '90235347')
(2, '88000003', '90053756')
(2, '88000003', '88300568')
(2, '88000003', '87258188')
(2, '88000003', '90067736')
(2, '88000003', '91000024')
(2, '88000003', '91365436')
(3, '88000006', '87124472')
(3, '88000006', '91304279')
(3, '88000006', '89369540')
(3, '88000006', '88269078')
(3, '88000006', '88217164')
(3, '88000006', '91171322')
(3, '88000006', '91103342')


Generating negs:   0%|          | 7/334070 [00:00<6:45:48, 13.72it/s]

(4, '88000007', '88089901')
(4, '88000007', '87063047')
(4, '88000007', '89199482')
(4, '88000007', '88063767')
(4, '88000007', '90029019')
(4, '88000007', '90278519')
(4, '88000007', '89377694')
(5, '88000008', '89116260')
(5, '88000008', '88011529')
(5, '88000008', '89015712')
(5, '88000008', '91306935')
(5, '88000008', '91346835')
(5, '88000008', '87323301')
(5, '88000008', '90138948')
(6, '88000009', '89383137')
(6, '88000009', '88132617')
(6, '88000009', '88202326')
(6, '88000009', '88037721')
(6, '88000009', '89069281')
(6, '88000009', '89203802')
(6, '88000009', '91327659')
(7, '88000010', '90257763')
(7, '88000010', '91020452')
(7, '88000010', '90028744')
(7, '88000010', '91237567')
(7, '88000010', '90198074')
(7, '88000010', '88279351')
(7, '88000010', '88004301')


Generating negs:   0%|          | 11/334070 [00:00<6:28:02, 14.35it/s]

...

Generating negs:   0%|          | 443/334070 [00:28<6:14:48, 14.84it/s]

(440, '88003931', '90088269')
(440, '88003931', '89229410')
(440, '88003931', '88127481')
(440, '88003931', '91302986')
(440, '88003931', '91074905')
(440, '88003931', '91319747')
(440, '88003931', '90071690')
(441, '88003932', '91189607')
(441, '88003932', '89047927')
(441, '88003932', '90327802')
(441, '88003932', '91003679')
(441, '88003932', '87299260')
(441, '88003932', '88090581')
(441, '88003932', '87175701')
(442, '88003933', '90100557')
(442, '88003933', '91328303')
(442, '88003933', '91118121')
(442, '88003933', '89012399')
(442, '88003933', '89209045')
(442, '88003933', '88287726')
(442, '88003933', '90091977')
(443, '88003934', '91305480')
(443, '88003934', '88173623')
(443, '88003934', '91028984')
(443, '88003934', '88065763')
(443, '88003934', '87143465')
(443, '88003934', '88327297')
(443, '88003934', '88301313')


Generating negs:   0%|          | 445/334070 [00:29<6:04:35, 15.25it/s]


(444, '88003938', '88312072')
(444, '88003938', '88033195')
(444, '88003938', '90167612')
(444, '88003938', '91369012')
(444, '88003938', '89290943')
(444, '88003938', '88244375')
(444, '88003938', '87303027')


KeyboardInterrupt: 